Let's implement logistic regression from scratch using numpy: 

- sigmoid, binary cross-entropy loss
- gradient descent update step. 
- train it on a small 2D dataset
- return learned weights

In [5]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

sigmoid(.7)

np.float64(0.6681877721681662)

In [11]:
def bce_loss(y_true, y_proba):
    eps = 1e-15
    y_proba = np.clip(y_proba, eps, 1 - eps)
    losses = -(y_true * np.log(y_proba) + (1 - y_true) * np.log(1 - y_proba))
    return np.mean(losses)

def mse_loss(y_true, y_proba):
    return np.mean((y_true - y_proba) ** 2)

In [15]:
n=100
threshold = 0.4
y_true = np.random.randint(0, 2, size=n)
y_proba = np.random.rand(n)
y_pred = (y_proba >= threshold).astype(int)

print("BCE: ",bce_loss(y_true, y_proba))
print("MSE: ", mse_loss(y_true, y_proba))

BCE:  0.9789957292104945
MSE:  0.3497915360836215


**About Sigmoid**

It is used as the final activation whenever the output is a binary probability - regardless of whether the model is logistic regression or a deep neural network.

**Binary classification**: last layer is one neuron with sigmoid → outputs a single probability → use BCE loss.

**Multi-class classification**: last layer is N neurons with **softmax** → outputs N probabilities that sum to 1 → use cross-entropy loss (the multi-class generalisation of BCE).

**Regression**: last layer has no activation (or linear) → outputs a raw number → use MSE loss.

So sigmoid isn't specific to logistic regression — logistic regression is just the simplest possible case: no hidden layers, one sigmoid output. A deep network for binary classification is the same idea with layers stacked before it.

**Inside the network** (hidden layers), sigmoid is mostly dead. ReLU (`max(0, x)`) replaced it because sigmoid's gradient vanishes for large/small inputs, making deep networks train slowly. You'll occasionally see GELU or SiLU (a smooth variant of ReLU) in modern transformers, but the principle is the same — sigmoid lives at the output for probabilities, not inside the network.

In [21]:
def logistic_regression(X, y, lr=0.1, epochs=1000):
    n, d = X.shape              # n samples, d features
    w = np.zeros(d)             # weights, one per feature
    b = 0                       # bias

    for epoch in range(epochs):
        # forward pass
        z = X @ w + b           # linear combination
        y_proba = sigmoid(z)    # squash to probabilities

        # compute loss (optional, for monitoring)
        loss = bce_loss(y, y_proba)

        # gradients
        error = y_proba - y     # shape (n,)
        dw = (1 / n) * X.T @ error   # shape (d,)
        db = (1 / n) * np.sum(error)  # scalar

        # update
        w -= lr * dw
        b -= lr * db

        if epoch % 200 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    return w, b

In [26]:
np.random.seed(42)
# two clusters in 2D
X = np.vstack([
    np.random.randn(50, 2) + [2, 2],   # class 1 centered at (2,2)
    np.random.randn(50, 2) + [-2, -2]   # class 0 centered at (-2,-2)
])
y = np.array([1]*50 + [0]*50)

w, b = logistic_regression(X, y, lr=0.1, epochs=1000)
print(f"Weights: {w}, Bias: {b}")

# predict
y_pred = (sigmoid(np.dot(X, w) + b) >= 0.5).astype(int)
print(f"Accuracy: {np.mean(y_pred == y):.2f}")

Epoch 0, Loss: 0.6931
Epoch 200, Loss: 0.0176
Epoch 400, Loss: 0.0103
Epoch 600, Loss: 0.0074
Epoch 800, Loss: 0.0059
Weights: [2.24584561 1.8532904 ], Bias: -0.04235699666044877
Accuracy: 1.00
